# The Lazy Book Report

Your professor has assigned a book report on "The Red-Headed League" by Arthur Conan Doyle. 

You haven't read the book. And out of stubbornness, you won't.

But you *have* learned NLP. Let's use it to answer the professor's questions without reading.

## Setup

First, let's fetch the text from Project Gutenberg and prepare it for analysis.

In [1]:
# Fetch and prepare text - RUN THIS CELL FIRST
import os
import urllib.request
import re

os.makedirs("output", exist_ok=True)

url = 'https://www.gutenberg.org/files/1661/1661-0.txt'
req = urllib.request.Request(url, headers={'User-Agent': 'Python-urllib'})
with urllib.request.urlopen(req, timeout=30) as resp:
    text = resp.read().decode('utf-8')

# Strip Gutenberg boilerplate
text = text.split('*** START OF')[1].split('***')[1]
text = text.split('*** END OF')[0]

# Extract "The Red-Headed League" story (it's the second story in the collection)
matches = list(re.finditer(r'THE RED-HEADED LEAGUE', text, re.IGNORECASE))
story_start = matches[1].end()
story_text = text[story_start:]
story_end = re.search(r'\n\s*III\.\s*\n', story_text)
story_text = story_text[:story_end.start()] if story_end else story_text

# Split into 3 sections by word count
words = story_text.split()[:4000]
section_size = len(words) // 3
sections = [
    ' '.join(words[:section_size]),
    ' '.join(words[section_size:2*section_size]),
    ' '.join(words[2*section_size:])
]

print(f"Story loaded: {len(words)} words in {len(sections)} sections")
print(f"Section sizes: {[len(s.split()) for s in sections]}")

Story loaded: 4000 words in 3 sections
Section sizes: [1333, 1333, 1334]


## Professor's Questions

Your professor wants you to answer 5 questions about the story. Let's use NLP to find the answers.

---

## Question 1: Writing Style

> "This text is from the 1890s. What makes it different from modern writing?"

**NLP Method:** Use preprocessing to compute text statistics. Tokenize the text and calculate:
- Vocabulary richness (unique words / total words)
- Average sentence length
- Average word length

**Hint:** Formal, literary writing typically shows higher vocabulary richness and longer sentences than modern casual text.

In [2]:
# Your code here: compute text statistics
# You'll need: import string, import re
# - Tokenize: remove punctuation, lowercase
# - Sentences: split on sentence-ending punctuation
# Calculate vocab_richness, avg_sentence_length, avg_word_length

import string
import re

tokens = story_text.translate(str.maketrans('', '', string.punctuation)).lower().split()
sentences = re.split(r'[.!?]+', story_text)

words = [w for w in tokens if w]
total_words = len(words)
unique_words = len(set(words))
vocab_richness = unique_words / total_words if total_words > 0 else 0

total_sentences = [s for s in sentences if s.strip()]
avg_sentence_length = total_words / len(total_sentences) if total_sentences else 0
avg_word_length = sum(len(w) for w in words) / total_words if total_words > 0 else 0

print(f"Vocabulary Richness: {vocab_richness:.4f}")
print(f"Average Sentence Length: {avg_sentence_length:.2f} words")
print(f"Average Word Length: {avg_word_length:.2f} characters")

Vocabulary Richness: 0.0997
Average Sentence Length: 14.55 words
Average Word Length: 4.19 characters


---

## Question 2: Main Characters

> "Who are the main characters in this story?"

**NLP Method:** Use Named Entity Recognition (NER) to extract PERSON entities.

**Hint:** Use spaCy's `en_core_web_sm` model. Process the text and filter entities where `ent.label_ == 'PERSON'`. Count how often each name appears.

In [3]:
# Your code here: extract PERSON entities using spaCy NER
# You'll need: import spacy, nlp = spacy.load("en_core_web_sm")

# When done, save your findings:
# with open("output/characters.txt", "w") as f:
#     for name in your_character_list:
#         f.write(f"{name}\n")

import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(story_text)
persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
# count how often each PERSON appears
counts = {name: persons.count(name) for name in set(persons)}

with open("output/characters.txt", "w") as f:
    for items in sorted(counts.items(), key=lambda x: x[1], reverse=True):
        f.write(f"{items}\n")

print(f"Extracted {len(set(persons))} unique PERSON entities.")



Extracted 262 unique PERSON entities.


---

## Question 3: Story Locations

> "Where does the story take place?"

**NLP Method:** Use Named Entity Recognition (NER) to extract location entities (GPE and LOC).

**Hint:** Filter entities where `ent.label_` is 'GPE' (geopolitical entity) or 'LOC' (location).

In [4]:
# Your code here: extract GPE and LOC entities using spaCy NER

# When done, save your findings:
# with open("output/locations.txt", "w") as f:
#     for place in your_locations_list:
#         f.write(f"{place}\n")

locations = set([ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC")])

with open("output/locations.txt", "w") as f:
    for place in locations:
        f.write(f"{place}\n")



---

## Question 4: Wilson's Business

> "What is Wilson's business?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's business.

**Hint:** Create a TF-IDF vectorizer, fit it on the 3 sections, then transform your query using the same vectorizer (`.transform()`, not `.fit_transform()` - you want to use the vocabulary learned from the sections). Find which section has the highest cosine similarity and read it to find the answer.

In [5]:
# Your code here: use TF-IDF similarity to find the relevant section
# You'll need: from sklearn.feature_extraction.text import TfidfVectorizer
#              from sklearn.metrics.pairwise import cosine_similarity

# When done, save your findings:
# with open("output/business.txt", "w") as f:
#     f.write("Wilson's business is: ...")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

verctorizer = TfidfVectorizer(stop_words='english')
matrix = verctorizer.fit_transform(sections)

search_term = "Wilson's business"
search_vec = verctorizer.transform([search_term])
cosine = cosine_similarity(search_vec, matrix).flatten()
best_section = sections[cosine.argmax()]

for line in best_section.splitlines():
   if "business" in line.lower():
      print(line)

with open("output/business.txt", "w") as f:
    f.write("Wilson's business is: a small pawnbroker's business at Coburg Square")
    

your fortunes. You will first make a note, Doctor, of the paper and the date.” “It is _The Morning Chronicle_ of April 27, 1890. Just two months ago.” “Very good. Now, Mr. Wilson?” “Well, it is just as I have been telling you, Mr. Sherlock Holmes,” said Jabez Wilson, mopping his forehead; “I have a small pawnbroker’s business at Coburg Square, near the City. It’s not a very large affair, and of late years it has not done more than just give me a living. I used to be able to keep two assistants, but now I only keep one; and I would have a job to pay him but that he is willing to come for half wages so as to learn the business.” “What is the name of this obliging youth?” asked Sherlock Holmes. “His name is Vincent Spaulding, and he’s not such a youth, either. It’s hard to say his age. I should not wish a smarter assistant, Mr. Holmes; and I know very well that he could better himself and earn twice what I am able to give him. But, after all, if he is satisfied, why should I put ideas in 

---

## Question 5: Wilson's Work Routine

> "What is Wilson's daily work routine for the League?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's work routine.

**Hint:** Similar to Question 4 - use TF-IDF to find the section that best matches your query about work routine. The answer includes what Wilson had to do and what eventually happened.

In [8]:
# Your code here: use TF-IDF similarity to find the relevant section

# When done, save your findings:
# with open("output/routine.txt", "w") as f:
#     f.write("Wilson's work routine: ...\n")
#     f.write("What happened: ...\n")

query = "Wilson daily work routine League"

search_vec = verctorizer.transform([query])
cosine = cosine_similarity(search_vec, matrix).flatten()
best_section = sections[cosine.argmax()]

print(best_section)

with open("output/routine.txt", "w") as f:
    f.write("Wilson's work routine: to be in the office for four hours every day, and to copy out the Encyclopaedia Britannica")
    f.write("What happened: after eight weeks, the League was disolved and Wilson's job ended")

any of the others, and he closed the door as we entered, so that he might have a private word with us. “‘This is Mr. Jabez Wilson,’ said my assistant, ‘and he is willing to fill a vacancy in the League.’ “‘And he is admirably suited for it,’ the other answered. ‘He has every requirement. I cannot recall when I have seen anything so fine.’ He took a step backward, cocked his head on one side, and gazed at my hair until I felt quite bashful. Then suddenly he plunged forward, wrung my hand, and congratulated me warmly on my success. “‘It would be injustice to hesitate,’ said he. ‘You will, however, I am sure, excuse me for taking an obvious precaution.’ With that he seized my hair in both his hands, and tugged until I yelled with the pain. ‘There is water in your eyes,’ said he as he released me. ‘I perceive that all is as it should be. But we have to be careful, for we have twice been deceived by wigs and once by paint. I could tell you tales of cobbler’s wax which would disgust you with